In [1]:
from sentence_transformers import SentenceTransformer

import numpy as np

c:\proyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
documentos =  [
    "El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático",
    "Solicitud de aumento de línea de crédito para cuenta empresarial",
    "El cliente no reconoce un cargo de $150 en su estado de cuenta",
    "Consulta sobre requisitos para abrir una cuenta de ahorros infantil",
    "Reclamo por doble cobro en transferencia interbancaria",
    "Solicitud de copia de estado de cuenta de los últimos 6 meses",
    "El cajero automático retuvo la tarjeta del cliente",
    "Consulta sobre tasa de interés para préstamo hipotecario",
    "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito",
    "Solicitud para cambiar el número de teléfono asociado a la cuenta",
    "El cliente reporta una transacción que no ha realizado con su tarjeta de crédito",
    "No está funcionando las transferencias ACH a las 08:00 am",
    "El usuario de banca en línea fue bloqueado y no puede entrar",
    "La app móvil se encuentra fuera de servicio",
    "El cliente solicita un préstam por consolidación de deudas",
    "La empresa ABC quiere cancelar su cuenta de cheques y abrir una nueva",
    "Se han incrementado las solicitudes de tarjeta de crédto para clientes con historial crediticio limitado",
    "Consulta para cancelar una tarjeta de crédito que ha sido robada",
    "El usuario de la app no puede ver el saldo de la tarjeta de crédito",
    "Solicitud para préstamo de pequeña empresa para capital de trabajo"
]

In [20]:
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embedding_docs = modelo.encode(documentos)

print(embedding_docs)  # (20, 384)

c:\proyecto\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\amyma\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\proyecto\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `hug

[[-0.00278847  0.09540856 -0.43310556 ... -0.14843068 -0.09072255
  -0.18465376]
 [ 0.08226307 -0.02862829 -0.31841314 ... -0.3605882  -0.2990758
  -0.06653975]
 [ 0.12315454  0.10069431 -0.08750795 ... -0.11724831 -0.15824169
  -0.2116466 ]
 ...
 [-0.1029171   0.2714748  -0.4644348  ... -0.18284103 -0.21163373
   0.11220942]
 [-0.19400737 -0.06343802 -0.26125324 ... -0.2359809  -0.17472564
  -0.17749882]
 [-0.08207162  0.00126006 -0.13292986 ... -0.15150867 -0.11735423
  -0.14535318]]


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
def buscar_documentos_similares(consulta, documentos, embedding_docs, modelo, top_k=3):

    # 1. Convierte la pregunta a embedding
    embedding_consulta = modelo.encode([consulta])

    # 2. Calcula la similitud entre todos los documentos y la consulta
    similitudes = cosine_similarity(embedding_consulta, embedding_docs)[0]

    # 3. Obtén los índices de los documentos más similares
    indices_top = np.argsort(similitudes)[::-1][:top_k]

    # 4. Devuelve los documentos más similares y sus puntuaciones de similitud
    resultados = [(documentos[i], similitudes[i]) for i in indices_top]

    return resultados

In [8]:
pregunta = "alguien uso mi tarjeta sin permiso"

resultados = buscar_documentos_similares(pregunta, documentos, embedding_docs, modelo, top_k=3)

for texto, score in resultados:
    print(f"Documento: {texto}\nSimilitud: {score:.4f}\n")

Documento: El cajero automático retuvo la tarjeta del cliente
Similitud: 0.5455

Documento: El usuario de la app no puede ver el saldo de la tarjeta de crédito
Similitud: 0.4449

Documento: El cliente no reconoce un cargo de $150 en su estado de cuenta
Similitud: 0.4370



In [21]:
import time
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

client = QdrantClient(host="localhost", port=6333)
COL = "tickets_banco"

# Crear colección 
if client.collection_exists(COL):
    client.delete_collection(COL)

client.create_collection(
    COL, vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

c:\proyecto\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:287: UserWarning: Qdrant client version 1.20.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


True

In [22]:
t0 = time.perf_counter()
client.upsert(COL, points=[
    PointStruct(id=i, vector=emb.tolist(), payload={"texto": doc})
    for i, (doc, emb) in enumerate(zip(documentos, embedding_docs))
])
print(f"Carga: {time.perf_counter() - t0:.3f} s")
print("Puntos en Qdrant:", client.count(COL).count)

Carga: 0.028 s
Puntos en Qdrant: 20


In [23]:
def buscar_qdrant(consulta, top_k=3):
    vector = modelo.encode(consulta).tolist()
    res = client.query_points(COL, query=vector, limit=top_k)
    return [(p.payload["texto"], p.score) for p in res.points]

for texto, score in buscar_qdrant(pregunta):
    print(f"Documento: {texto}\nSimilitud: {score:.4f}\n")

Documento: El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
Similitud: 0.6078

Documento: El cajero automático retuvo la tarjeta del cliente
Similitud: 0.6022

Documento: Consulta para cancelar una tarjeta de crédito que ha sido robada
Similitud: 0.5857



In [24]:
preguntas_prueba = [
    ("alguien uso mi tarjeta sin permiso", "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito"),
    ("me quedé sin poder entrar a la banca en línea", "El usuario de banca en línea fue bloqueado y no puede entrar"),
    ("quiero un crédito para mi negocio", "Solicitud para préstamo de pequeña empresa para capital de trabajo"),
    ("el cajero se tragó mi tarjeta", "El cajero automático retuvo la tarjeta del cliente"),
    ("necesito mis estados de cuenta anteriores", "Solicitud de copia de estado de cuenta de los últimos 6 meses"),
]

In [25]:
def evaluar(buscar_fn, preguntas, top_k=3):
    aciertos = 0
    tiempos = []
    for pregunta, esperado in preguntas:
        t0 = time.perf_counter()
        resultados = buscar_fn(pregunta, top_k)
        tiempos.append(time.perf_counter() - t0)
        textos = [r[0] for r in resultados]
        if esperado in textos:
            aciertos += 1
    print(f"Aciertos en top-{top_k}: {aciertos}/{len(preguntas)}")
    print(f"Latencia promedio: {sum(tiempos)/len(tiempos)*1000:.1f} ms")

evaluar(buscar_qdrant, preguntas_prueba)

Aciertos en top-3: 4/5
Latencia promedio: 37.9 ms


In [26]:
import numpy as np

vectores_q = [modelo.encode(p).tolist() for p, _ in preguntas_prueba]

tiempos = []
for _ in range(50):
    for v in vectores_q:
        t0 = time.perf_counter()
        client.query_points(COL, query=v, limit=3)
        tiempos.append((time.perf_counter() - t0) * 1000)

print(f"Latencia (solo Qdrant): promedio {np.mean(tiempos):.1f} ms | p95 {np.percentile(tiempos, 95):.1f} ms")

Latencia (solo Qdrant): promedio 15.1 ms | p95 26.1 ms


In [29]:
categorias = [
    "tarjetas", "prestamos", "fraude", "cuentas", "transferencias",
    "cuentas", "tarjetas", "prestamos", "fraude", "cuentas",
    "fraude", "canales_digitales", "canales_digitales", "canales_digitales", "prestamos",
    "cuentas", "tarjetas", "tarjetas", "canales_digitales", "prestamos",
]

client.upsert(COL, points=[
    PointStruct(id=i, vector=emb.tolist(), payload={"texto": doc, "categoria": cat})
    for i, (doc, emb, cat) in enumerate(zip(documentos, embedding_docs, categorias))
])

client.create_payload_index(COL, field_name="categoria", field_schema="keyword")

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [30]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

v = modelo.encode("alguien uso mi tarjeta sin permiso").tolist()

print("SIN filtro:")
for p in client.query_points(COL, query=v, limit=3).points:
    print(f"  {p.score:.3f} | {p.payload['categoria']} | {p.payload['texto']}")

filtro = Filter(must=[FieldCondition(key="categoria", match=MatchValue(value="fraude"))])
print("\nCON filtro categoria=fraude:")
for p in client.query_points(COL, query=v, query_filter=filtro, limit=3).points:
    print(f"  {p.score:.3f} | {p.payload['categoria']} | {p.payload['texto']}")

SIN filtro:
  0.608 | tarjetas | El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
  0.602 | tarjetas | El cajero automático retuvo la tarjeta del cliente
  0.586 | tarjetas | Consulta para cancelar una tarjeta de crédito que ha sido robada

CON filtro categoria=fraude:
  0.576 | fraude | El cliente reporta una transacción que no ha realizado con su tarjeta de crédito
  0.491 | fraude | Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito
  0.411 | fraude | El cliente no reconoce un cargo de $150 en su estado de cuenta


In [31]:
from qdrant_client.models import VectorParams, Distance, PointStruct

BENCH = "bench"
N = 20000

if client.collection_exists(BENCH):
    client.delete_collection(BENCH)
client.create_collection(BENCH, vectors_config=VectorParams(size=384, distance=Distance.COSINE))

rng = np.random.default_rng(42)
datos = rng.normal(size=(N, 384)).astype("float32")

t0 = time.perf_counter()
for i in range(0, N, 1000):
    client.upsert(BENCH, points=[
        PointStruct(id=j, vector=datos[j].tolist()) for j in range(i, min(i + 1000, N))
    ])
print(f"Carga de {N} vectores: {time.perf_counter() - t0:.2f} s")

consulta = rng.normal(size=384).tolist()
tiempos = []
for _ in range(100):
    t0 = time.perf_counter()
    client.query_points(BENCH, query=consulta, limit=10)
    tiempos.append((time.perf_counter() - t0) * 1000)
print(f"Latencia con {N} vectores: promedio {np.mean(tiempos):.1f} ms | p95 {np.percentile(tiempos, 95):.1f} ms")

Carga de 20000 vectores: 4.23 s
Latencia con 20000 vectores: promedio 17.1 ms | p95 30.3 ms


In [32]:
from qdrant_client.models import SearchParams

aciertos, total = 0, 0
for _ in range(50):
    q = rng.normal(size=384).tolist()
    aprox = {p.id for p in client.query_points(BENCH, query=q, limit=10).points}
    exacta = {p.id for p in client.query_points(
        BENCH, query=q, limit=10, search_params=SearchParams(exact=True)).points}
    aciertos += len(aprox & exacta)
    total += 10
print(f"Recall@10 (HNSW vs exacta): {aciertos/total:.2%}")

Recall@10 (HNSW vs exacta): 100.00%


In [33]:
for p_, esperado in preguntas_prueba:
    top = buscar_qdrant(p_, 3)
    ok = esperado in [t for t, _ in top]
    print("✅" if ok else "❌", p_)
    if not ok:
        print("   Esperado:", esperado)
        for t, s in top:
            print(f"   {s:.3f} | {t}")

❌ alguien uso mi tarjeta sin permiso
   Esperado: Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito
   0.608 | El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
   0.602 | El cajero automático retuvo la tarjeta del cliente
   0.586 | Consulta para cancelar una tarjeta de crédito que ha sido robada
✅ me quedé sin poder entrar a la banca en línea
✅ quiero un crédito para mi negocio
✅ el cajero se tragó mi tarjeta
✅ necesito mis estados de cuenta anteriores
